# In-Class Exercise: Model Validation & Causal Inference

**Course:** ECON6083 - Machine Learning in Economics  
**Topic:** Cross-Validation, Metrics, and Causal Inference  
**Time:** 25 minutes  

This exercise covers the three main parts of Lecture 4:
1. Cross-Validation & Hyperparameter Tuning
2. Evaluation Metrics
3. Causal Inference (IPW & AIPW)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')


---

## Part 1: Cross-Validation & Model Selection (8 minutes)

We use a credit default dataset with the following variables:
- `income`, `debt_ratio`, `credit_score`: Features
- `default`: Binary outcome (1 = defaulted)

In [ ]:
# Generate synthetic credit data
np.random.seed(42)
n = 1000

income = np.random.lognormal(10, 0.5, n)
debt_ratio = np.random.beta(2, 5, n)
credit_score = np.random.normal(650, 80, n).clip(300, 850)

# Default probability (non-linear)
prob_default = 1 / (1 + np.exp(-(-5 + 2*debt_ratio - 0.01*credit_score + 0.001*income)))
default = np.random.binomial(1, prob_default)

df = pd.DataFrame({
    'income': income,
    'debt_ratio': debt_ratio,
    'credit_score': credit_score,
    'default': default
})

X = df[['income', 'debt_ratio', 'credit_score']]
y = df['default']

print(f"Default rate: {y.mean():.3f}")
df.head()


### Exercise 1.1: Compare CV vs Single Split

Train a Random Forest and compare:
1. Single train-test split (80/20) with `random_state=42`
2. 5-fold cross-validation

**Tasks:**
1. Run both methods and compare the AUC scores
2. **Print out ALL 5 CV fold scores** (not just the mean) — use `print(cv_scores)`
3. Observe: Are all 5 fold scores the same? What's the range (max - min)?
4. Answer the discussion questions below

**Discussion Questions:**
- Q1: Is the CV mean always HIGHER than single split? Can CV be lower?
- Q2: What does the +/- (standard deviation) tell us about the uncertainty?
- Q3: Why is CV more reliable even if it gives a lower score?


In [ ]:
# Single train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=____, random_state=42, stratify=____)  # TODO: 0.2 test size; stratify on y

rf = RandomForestClassifier(n_estimators=____, random_state=42)  # TODO: set to 50 trees
rf.fit(X_train, y_train)
single_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

# 5-fold CV
cv_scores = cross_val_score(____, X, y, cv=5, scoring='roc_auc')  # TODO: pass the trained rf model

print(f"Single split AUC: {single_auc:.4f}")
print(f"5-fold CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# Print all 5 fold scores and calculate range
print("\nAll 5 fold scores:", cv_scores)
print(f"Range (max - min): {cv_scores.max() - cv_scores.min():.4f}")
print(f"Fold scores vary from {cv_scores.min():.4f} to {cv_scores.max():.4f}")


---

## Part 2: Evaluation Metrics (8 minutes)

Using the trained model, calculate different metrics and understand the trade-offs.

### Exercise 2.1: Confusion Matrix & Metrics

Calculate and interpret:
- **Precision**: Of predicted defaults, how many actually defaulted?
- **Recall**: Of actual defaults, how many did we catch?

**Question:** For a bank, which metric is more important? Why?

In [ ]:
# Get predictions
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

# Calculate confusion matrix
cm = confusion_matrix(____, ____)  # TODO: pass y_test and y_pred
print("Confusion Matrix:")
print(cm)

# Calculate precision and recall
precision = precision_score(____, ____, zero_division=0)  # TODO: pass y_test and y_pred
recall = recall_score(____, ____, zero_division=0)  # TODO: pass y_test and y_pred

print(f"\nPrecision: {precision:.3f}")
print(f"Recall: {recall:.3f}")


### Exercise 2.2: Threshold Selection

The default threshold is 0.5. Try different thresholds and see how precision/recall change.

In [ ]:
# Try different thresholds
thresholds = [0.3, 0.5, 0.7]

for thresh in thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    prec = precision_score(y_test, y_pred_thresh, zero_division=0)
    rec = recall_score(y_test, y_pred_thresh, zero_division=0)
    print(f"Threshold {thresh}: Precision={prec:.3f}, Recall={rec:.3f}")


**Discussion:** 
- At threshold 0.3: High recall, lower precision
- At threshold 0.7: High precision, lower recall

What threshold would you choose for:
- **Conservative bank** (avoid false alarms)?
- **Aggressive bank** (don't miss any defaulters)?

---

## Part 3: Causal Inference - IPW (9 minutes)

Now we simulate a job training program (LaLonde-style) to estimate causal effects.

In [ ]:
# Generate job training data
np.random.seed(42)
n = 1000

age = np.random.normal(30, 8, n)
education = np.random.poisson(12, n)

# Selection into treatment (non-random!)
prob_treat = 1 / (1 + np.exp(-(-2.0 + 0.15*age - 0.25*education)))
treat = np.random.binomial(1, prob_treat)

# True ATE = $1000
earnings = 15000 + 400*education + 1000*treat + np.random.normal(0, 2500, n)

causal_df = pd.DataFrame({
    'age': age,
    'education': education,
    'treat': treat,
    'earnings': earnings
})

print(f"Treatment rate: {treat.mean():.3f}")
print(f"\nRaw difference: ${causal_df.groupby('treat')['earnings'].mean().diff().iloc[1]:.2f}")
print("(True ATE = $1000)")


### Exercise 3.1: Naive vs IPW Estimate

Compare:
1. **Naive difference in means** (simple comparison)
2. **IPW estimate** (inverse propensity weighting)

The IPW formula:
$$\hat{\tau}_{IPW} = \frac{1}{n}\sum_{i=1}^n \left[\frac{D_i Y_i}{\hat{p}(X_i)} - \frac{(1-D_i)Y_i}{1-\hat{p}(X_i)}\right]$$

In [ ]:
# Step 1: Estimate propensity scores
X_ps = causal_df[['age', 'education']]
ps_model = LogisticRegression().fit(X_ps, causal_df['treat'])
causal_df['propensity'] = ps_model.predict_proba(X_ps)[:, 1]

# Clip to avoid division by zero
causal_df['propensity'] = causal_df['propensity'].clip(0.05, 0.95)

print("Propensity score distribution:")
print(causal_df['propensity'].describe())


In [ ]:
# Step 2: Calculate IPW estimate
D = causal_df['treat']
Y = causal_df['earnings']
p = causal_df['propensity']

# Implement IPW formula
treated_term = ____  # TODO: IPW treated term: D * Y / p
control_term = ____  # TODO: IPW control term: (1-D) * Y / (1-p)

ipw_ate = np.mean(treated_term - control_term)

print(f"\nNaive estimate: ${causal_df.groupby('treat')['earnings'].mean().diff().iloc[1]:.2f}")
print(f"IPW estimate: ${ipw_ate:.2f}")
print(f"True ATE: $1000")
print(f"\nIPW error: ${abs(ipw_ate - 1000):.2f}")


### Exercise 3.2: Understanding the Bias

**Questions to discuss:**

1. Why is the naive estimate biased? (Hint: Look at who gets treated)
2. How does IPW correct for this bias?
3. What would happen if we didn't clip propensity scores at 0.05 and 0.95?

In [ ]:
# Examine selection bias
print("Characteristics by treatment status:")
print(causal_df.groupby('treat')[['age', 'education']].mean())


### Exercise 3.3: AIPW - Augmented IPW (Bonus)

**What is AIPW?**

AIPW = IPW + Regression Correction

Think of it as having **two insurance policies**:
1. **IPW**: Corrects for selection bias using propensity scores
2. **Outcome regression**: Predicts what income would be with/without treatment

**AIPW combines both**: If one is wrong, the other can still save you!

---

#### Intuitive Explanation

Imagine you want to estimate the effect of a job training program:

**IPW only**: "John participated and his income is $50k. He had 20% chance of participating, so his weighted contribution is $50k / 0.2 = $250k."

→ Problem: If someone's probability is 1% or 99%, weights become extreme!

**AIPW**: "I predict John would earn $45k with training. He actually earned $50k, so my prediction was off by $5k. I'll add ($5k / 0.2) = $25k as a correction."

→ Result: $45k (prediction) + $25k (correction) = $70k

The regression prediction ($45k) is more stable, and the IPW part only corrects the **error** in prediction.

In [ ]:
# AIPW Implementation

# Step 1: Train outcome models for treated and control groups
# mu1(X) = E[Y|X, D=1]  - expected income if treated
# mu0(X) = E[Y|X, D=0]  - expected income if not treated

# Model for treated group
X_treat = causal_df[causal_df['treat']==1][['age', 'education']]
y_treat = causal_df[causal_df['treat']==1]['earnings']
model_treat = LinearRegression().fit(X_treat, y_treat)

# Model for control group  
X_control = causal_df[causal_df['treat']==0][['age', 'education']]
y_control = causal_df[causal_df['treat']==0]['earnings']
model_control = LinearRegression().fit(X_control, y_control)

# Predict potential outcomes for everyone
X_all = causal_df[['age', 'education']]
causal_df['mu1'] = model_treat.predict(X_all)  # Predicted income if treated
causal_df['mu0'] = model_control.predict(X_all)  # Predicted income if not treated

print("Example predictions (first 3 people):")
print(causal_df[['treat', 'earnings', 'mu1', 'mu0']].head(3))


In [ ]:
# Step 2: Calculate AIPW estimate

D = causal_df['treat'].values
Y = causal_df['earnings'].values
p = causal_df['propensity'].values
mu1 = causal_df['mu1'].values
mu0 = causal_df['mu0'].values

# AIPW formula:
# For treated: mu1 + D*(Y - mu1)/p
# For control: mu0 + (1-D)*(Y - mu0)/(1-p)

term1 = mu1 + D * (Y - mu1) / p       # AIPW term for treated potential outcome
term0 = mu0 + (1 - D) * (Y - mu0) / (1 - p)  # AIPW term for control potential outcome

aipw_ate = np.mean(term1 - term0)

# Compare all estimates
naive_ate = causal_df.groupby('treat')['earnings'].mean().diff().iloc[1]
ipw_ate = np.mean(D * Y / p - (1 - D) * Y / (1 - p))

print("="*50)
print("Comparison of Estimates (True ATE = $1000)")
print("="*50)
print(f"Naive estimate:  ${naive_ate:8.2f}  (Error: ${abs(naive_ate-1000):.2f})")
print(f"IPW estimate:    ${ipw_ate:8.2f}  (Error: ${abs(ipw_ate-1000):.2f})")
print(f"AIPW estimate:   ${aipw_ate:8.2f}  (Error: ${abs(aipw_ate-1000):.2f})")
print("="*50)


#### Understanding AIPW Results

**Why is AIPW often better?**

1. **More stable**: The regression prediction (`mu1`, `mu0`) provides a "baseline" that does not depend on extreme weights
2. **Corrects errors**: The IPW part only adjusts for prediction errors (`Y - mu1`), not the full outcome
3. **Double robustness**: If either the propensity score model OR the outcome model is correct, AIPW is consistent!

**Visual intuition:**

```
IPW only:   [Extreme weights dominate] -> Unstable, high variance

AIPW:       [Regression prediction]    -> Stable baseline
              + [Small IPW correction]  -> Adjusts for selection
            
            -> More accurate!
```

**When to use AIPW?**
- When you have extreme propensity scores (close to 0 or 1)
- When you believe your outcome model is reasonably good
- When you want the extra "insurance" of double robustness

---

## Summary: Key Takeaways

| Part | Key Concept | Practical Skill |
|:---|:---|:---|
| **1. CV** | Cross-validation reduces variance | `cross_val_score` |
| **2. Metrics** | Precision/Recall trade-off | Threshold selection |
| **3. Causal** | Selection bias & IPW/AIPW | Propensity score weighting |

**Bottom line:** Good prediction != Good causal inference. IPW helps but requires careful implementation. AIPW provides extra robustness by combining propensity weighting with outcome regression.